In [49]:
from pathlib import Path
from typing import Any

import optuna
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

In [50]:
Path.cwd()

PosixPath('/Users/cube/source/dhbw/exploration/src2')

In [51]:
root_dir = Path.cwd().parent
temp_dir = root_dir / ".temp"
assert temp_dir.exists()

temp_dir

PosixPath('/Users/cube/source/dhbw/exploration/.temp')

In [52]:
dataset_train_path = temp_dir / "vehicles_train.csv"
dataset_test_path = temp_dir / "vehicles_test.csv"
dataset_train_path, dataset_test_path

(PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_train.csv'),
 PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_test.csv'))

In [53]:
images_path = Path.cwd() / ".." / "charged-ieee" / "images"
images_path

PosixPath('/Users/cube/source/dhbw/exploration/src2/../charged-ieee/images')

Globals

In [54]:
RNG = 99

# Load

In [55]:
df_train, df_test = pd.read_csv(dataset_train_path), pd.read_csv(dataset_test_path)
df_train.shape, df_test.shape

((279106, 70), (93036, 70))

# Prepare

In [56]:
col_label = "price"
cols_features = list(set(df_train.columns) - {col_label})
col_label, len(cols_features)

('price', 69)

In [57]:
X_train, y_train = df_train[cols_features], df_train[col_label]
X_test, y_test = df_test[cols_features], df_test[col_label]
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((279106, 69), (279106,), (93036, 69), (93036,))

# Train

### Common

In [58]:
def suggest_isolation_forest_params(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "iso_n_estimators": trial.suggest_int("iso_n_estimators", 50, 200),
        "iso_contamination": "auto"
        if trial.suggest_categorical("iso_contamination_mode", ["auto", "manual"])
        == "auto"
        else trial.suggest_float("iso_contamination", 0.01, 0.5),
    }

In [59]:
def new_isolation_forest(model_params, **params) -> IsolationForest:
    return IsolationForest(
        n_estimators=model_params["iso_n_estimators"],
        contamination="auto"
        if model_params["iso_contamination"] == "auto"
        else model_params["iso_contamination"],
        random_state=RNG,
        **params,
    )

In [60]:
def fit_with_isolation_forest(model, X_in, y_in, model_params):
    model_iso = new_isolation_forest(model_params, n_jobs=-1)
    model_iso.fit(X_in)
    inlier_mask = model_iso.predict(X_in) == 1
    model.fit(X_in[inlier_mask], y_in[inlier_mask])

### Prefiltered Train Data

In [61]:
isolation_forest_params = {
    "iso_n_estimators": 157,
    "iso_contamination_mode": "manual",
    "iso_contamination": 0.4662846821308693,
}
isolation_forest = new_isolation_forest(isolation_forest_params)
isolation_forest.fit(X_train)
inlier_mask = isolation_forest.predict(X_train) == 1
X_train_inliers, y_train_inliers = X_train[inlier_mask], y_train[inlier_mask]
X_train_inliers.shape, y_train_inliers.shape

((148963, 69), (148963,))

### Model Dummy (Median)

In [27]:
model_dummy = DummyRegressor(strategy="median")
model_dummy.fit(X_train, y_train)
y_pred_dummy = model_dummy.predict(X_test)
mae_dummy, rmse_dummy, r2_dummy = (
    mean_absolute_error(y_test, y_pred_dummy),
    root_mean_squared_error(y_test, y_pred_dummy),
    r2_score(y_test, y_pred_dummy)
)
mae_dummy, rmse_dummy, r2_dummy

(10972.87008254869, 14264.327880967614, -0.04763379647935784)

### Model Linear

In [62]:
model_linear = LinearRegression(n_jobs=-1)
model_linear.fit(X_train_inliers, y_train_inliers)
y_pred_linear = model_linear.predict(X_test)
mae_linear, rmse_linear, r2_linear = (
    mean_absolute_error(y_test, y_pred_linear),
    root_mean_squared_error(y_test, y_pred_linear),
    r2_score(y_test, y_pred_linear),
)
mae_linear, rmse_linear, r2_linear

(6279.6243405749765, 9219.203040238026, 0.5623835549365597)

### Model RandomForest

In [18]:
def new_model_rft(model_params, **params) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=model_params["rf_n_estimators"],
        max_depth=model_params["rf_max_depth"],
        min_samples_split=model_params["rf_min_samples_split"],
        min_samples_leaf=model_params["rf_min_samples_leaf"],
        random_state=RNG,
        **params,
    )

In [19]:
def objective_model_rft(trial):
    # 1) Outlier filter (IsolationForest)
    # iso = new_isolation_forest(
    #     suggest_isolation_forest_params(trial),
    #     n_jobs=-1,
    # )
    # inlier_mask = iso.fit_predict(X_train) == 1
    # X_in, y_in = X_train[inlier_mask], y_train[inlier_mask]
    X_in, y_in = X_train_inliers, y_train_inliers

    # 2) Regressor (RandomForest)
    rf = new_model_rft(
        {
            "rf_n_estimators": trial.suggest_int("rf_n_estimators", 100, 500),
            "rf_max_depth": trial.suggest_int("rf_max_depth", 4, 24),
            "rf_min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 20),
            "rf_min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 10),
        },
        n_jobs=-1,
    )

    cv = KFold(n_splits=7, shuffle=True, random_state=RNG)
    mae = -cross_val_score(
        rf, X_in, y_in, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1
    ).mean()
    return mae  # minimize MAE

In [20]:
study_rft = optuna.create_study(direction="minimize")
study_rft.optimize(objective_model_rft, n_trials=50)
model_rft_params = study_rft.best_params
model_rft_params

[I 2026-04-20 00:37:35,072] A new study created in memory with name: no-name-36ca35c4-2137-4d1d-ab79-fdeca3251f64
[I 2026-04-20 00:39:13,398] Trial 0 finished with value: 4416.990335691354 and parameters: {'rf_n_estimators': 398, 'rf_max_depth': 8, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 7}. Best is trial 0 with value: 4416.990335691354.
[I 2026-04-20 00:41:23,612] Trial 1 finished with value: 3063.765971107129 and parameters: {'rf_n_estimators': 308, 'rf_max_depth': 17, 'rf_min_samples_split': 9, 'rf_min_samples_leaf': 1}. Best is trial 1 with value: 3063.765971107129.
[I 2026-04-20 00:44:48,199] Trial 2 finished with value: 2955.572175477856 and parameters: {'rf_n_estimators': 469, 'rf_max_depth': 21, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 6}. Best is trial 2 with value: 2955.572175477856.
[I 2026-04-20 00:45:30,280] Trial 3 finished with value: 3518.5625405371343 and parameters: {'rf_n_estimators': 114, 'rf_max_depth': 13, 'rf_min_samples_split': 2, 'rf_min_sam

KeyboardInterrupt: 

In [63]:
model_rft_params = {
    "rf_n_estimators": 389,
    "rf_max_depth": 24,
    "rf_min_samples_split": 10,
    "rf_min_samples_leaf": 3,
}

In [64]:
model_rft = new_model_rft(model_rft_params, n_jobs=-1)
model_rft.fit(X_train_inliers, y_train_inliers)
y_pred_rft = model_rft.predict(X_test)
mae_rft, rmse_rft, r2_rft = (
    mean_absolute_error(y_test, y_pred_rft),
    root_mean_squared_error(y_test, y_pred_rft),
    r2_score(y_test, y_pred_rft),
)
mae_rft, rmse_rft, r2_rft

(3866.6406863167695, 6609.650838256266, 0.7750614401490336)